In [4]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_11 = 5 * 50

def create_z_11(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_11)
        for _, row in branch_data.iterrows()
    ]

def create_y_11(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_11)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_11(branch_data, z_11, y_11):
    z_11 = np.array(z_11)
    y_11 = np.array(y_11)
    gammaL_11 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_11[i] * y_11[i])
        gammaL_11.append(gammaL)
    return gammaL_11

z_11 = create_z_11(branch_data)
y_11 = create_y_11(branch_data)
gammaL_11 = create_gammaL_11(branch_data, z_11, y_11)

def Z0_11(z_11, y_11):
    return np.sqrt(np.array(z_11) / np.array(y_11))

Z0_11 = Z0_11(z_11, y_11)
def ABCD_11(gammaL_11, Z0_11):
    A = np.cosh(gammaL_11)
    B = Z0_11 * np.sinh(gammaL_11)
    C = (1 / Z0_11) * np.sinh(gammaL_11)
    D = np.cosh(gammaL_11)
    return A, B, C, D
A_11, B_11, C_11, D_11 = ABCD_11(gammaL_11, Z0_11)
ABCD_11_df = pd.DataFrame({
    'A_11': A_11,
    'B_11': B_11,
    'C_11': C_11,
    'D_11': D_11
})
ABCD_11_df.to_csv("ABCD_11.csv", index=False)

In [6]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_11.csv")

# Clean column names (optional, in case there are hidden spaces)
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_11', 'B_11', 'C_11', 'D_11']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'From Bus Number': row['From Bus  Number'],
            'To Bus Number': row['To Bus  Number'],
            'A': row['A_11'],
            'B': row['B_11'],
            'C': row['C_11'],
            'D': row['D_11']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_11', 'B_11', 'C_11', 'D_11']]

    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_11'], row['B_11'], row['C_11'], row['D_11']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'From Bus Number': group.iloc[0]['From Bus  Number'],
        'To Bus Number': group.iloc[0]['To Bus  Number'],
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus  Number', 'To Bus  Number'])
    .apply(combine_parallel)
    .reset_index(drop=True)
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined.csv", index=False)

print(ABCD_combined.head())


   From Bus Number   To Bus Number                   A                    B  \
0   2135.0+   0.0j  2220.0+   0.0j  0.930616+0.002070j  0.145142+10.087491j   
1   2135.0+   0.0j  2400.0+   0.0j -0.018558+0.030480j -0.026429+47.055113j   
2   2135.0+   0.0j  2970.0+   0.0j -0.226617-0.016158j -0.110736-28.628424j   
3   2220.0+   0.0j  2225.0+   0.0j  0.957358-0.007798j  0.336360- 3.796587j   
4   2220.0+   0.0j  2230.0+   0.0j -0.007088-0.012550j -0.002380-22.755487j   

                    C                   D  
0  0.000191+0.013282j  0.930616+0.002070j  
1 -0.000012+0.021264j -0.018558+0.030480j  
2 -0.000128-0.033146j -0.226617-0.016158j  
3  0.001968-0.022175j  0.957358-0.007798j  
4 -0.000003-0.043950j -0.007088-0.012550j  


C:\Users\User\AppData\Local\Temp\ipykernel_20268\2143568679.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)


In [7]:
import pandas as pd
import numpy as np

# Example Z0 calculation (if you already have Z0_11 column)

Z0 = np.abs(Z0_11)  # or define a constant if same for all lines
Y0 = 1 / Z0

def abcd2s(ABCD_combined, Z0, Y0):
    S_data = []
    for _, row in ABCD_combined.iterrows():
        # Extract ABCD parameters
        AA = row["A"]
        BB = row["B"]
        CC = row["C"]
        DD = row["D"]

        # Compute S-parameters
        denom1 = (AA + BB * Y0 + CC * Z0 + DD)
        denom2 = (AA + BB * Y0 + CC * Z0 - DD)

        S11 = (AA + BB * Y0 - CC * Z0 - DD) / denom1
        S12 = 2 * (AA * DD - BB * CC) / denom1
        S21 = 2 / denom1
        S22 = (-AA + BB * Y0 - CC * Z0 + DD) / denom1

        S_data.append({
            "From Bus": row["From Bus Number"],
            "To Bus": row["To Bus Number"],
            "S11": S11,
            "S12": S12,
            "S21": S21,
            "S22": S22
        })

    return pd.DataFrame(S_data)

# Run conversion
S_params = abcd2s(ABCD_combined, Z0, Y0)

print(S_params.head())


         From Bus          To Bus  \
0  2135.0+   0.0j  2220.0+   0.0j   
1  2135.0+   0.0j  2400.0+   0.0j   
2  2135.0+   0.0j  2970.0+   0.0j   
3  2220.0+   0.0j  2225.0+   0.0j   
4  2220.0+   0.0j  2230.0+   0.0j   

                                                 S11  \
0  [(-0.11919635457368248-0.23447158507455645j), ...   
1  [(-0.15247784159154693+0.0027061231243192163j)...   
2  [(-0.5300731240267984-0.10090485833340386j), (...   
3  [(-0.2943589892145571+0.37879472993301205j), (...   
4  [(-0.7025312228860462-0.003489082513618183j), ...   

                                                 S12  \
0  [(0.8601413934677067-0.4218324142624641j), (0....   
1  [(-0.017568638230441604-0.9581267167203658j), ...   
2  [(-0.15746076237566384+0.8102157458500922j), (...   
3  [(0.6966982423869935+0.44860976541224107j), (0...   
4  [(-0.0035192859844620813+0.6991960245146536j),...   

                                                 S21  \
0  [(0.860141393467707-0.42183241426246426j), (

In [13]:
import pandas as pd
import numpy as np

def connection_matrix(ABCD_combined):
    # Clean and extract unique buses
    buses = pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel())
    
    # Sort buses for readability (optional)
    buses = np.sort(buses)
    
    # Map bus numbers to index
    bus_index = {bus: idx for idx, bus in enumerate(buses)}
    n = len(buses)

    # Initialize connection matrix
    connection_matrix = np.zeros((n, n), dtype=int)

    # Build the adjacency (connection) matrix
    for _, row in ABCD_combined.iterrows():
        from_idx = bus_index[row['From Bus Number']]
        to_idx = bus_index[row['To Bus Number']]
        connection_matrix[from_idx, to_idx] = 1
        connection_matrix[to_idx, from_idx] = 1  # undirected

    # Create DataFrame with formatted bus names
    bus_labels = [f"{bus.real:+.1f}+{bus.imag:+.1f}j" if isinstance(bus, complex) else f"{bus:.1f}+0.0j" for bus in buses]
    connection_df = pd.DataFrame(connection_matrix, index=bus_labels, columns=bus_labels)
    return connection_df

# Create the connection matrix DataFrame
connection_matrix_df = connection_matrix(ABCD_combined)

# Convert to NumPy array
conn_np = connection_matrix_df.to_numpy()

print(conn_np)
print("Shape:", conn_np.shape)

[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1]
 [1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1]
 [0 1 0 0 

In [ ]:
#def connection_matrix_with_labels(ABCD_combined):
    # Extract unique buses
    buses = pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel())
    buses = np.sort(buses)
    
    # Map bus numbers to simple labels
    from_labels = {bus: f"a{i+1}" for i, bus in enumerate(buses)}
    to_labels = {bus: f"b{i+1}" for i, bus in enumerate(buses)}
    
    # Map bus numbers to index
    bus_index = {bus: idx for idx, bus in enumerate(buses)}
    n = len(buses)
    
    # Initialize connection matrix
    conn_matrix = np.zeros((n, n), dtype=int)
    
    # Fill matrix
    for _, row in ABCD_combined.iterrows():
        from_idx = bus_index[row['From Bus Number']]
        to_idx = bus_index[row['To Bus Number']]
        conn_matrix[from_idx, to_idx] = 1
        conn_matrix[to_idx, from_idx] = 1  # undirected
    
    # Create DataFrame with new labels
    from_bus_labels = [from_labels[bus] for bus in buses]
    to_bus_labels = [to_labels[bus] for bus in buses]
    conn_df = pd.DataFrame(conn_matrix, index=from_bus_labels, columns=to_bus_labels)
    
    return conn_df

# Example usage:
#connection_matrix_df = connection_matrix_with_labels(ABCD_combined)
#print(connection_matrix_df)

     b1  b2  b3  b4  b5  b6  b7  b8  b9  b10  ...  b17  b18  b19  b20  b21  \
a1    0   1   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a2    1   0   0   1   1   0   0   0   0    0  ...    0    0    1    0    1   
a3    0   0   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a4    0   1   0   0   0   0   0   0   0    0  ...    0    0    0    0    0   
a5    0   1   0   0   0   1   0   0   0    0  ...    0    0    0    0    0   
a6    0   0   0   0   1   0   0   1   0    0  ...    0    0    0    0    0   
a7    0   0   0   0   0   0   0   0   1    0  ...    0    0    0    0    0   
a8    0   0   0   0   0   1   0   0   0    0  ...    0    0    0    0    0   
a9    0   0   0   0   0   0   1   0   0    1  ...    0    0    0    0    0   
a10   0   0   0   0   0   0   0   0   1    0  ...    0    0    0    0    0   
a11   0   0   1   0   0   0   0   0   0    0  ...    0    0    1    0    0   
a12   0   0   0   0   0   0   0   0   0    0  ...    0    0    0

In [14]:
buses = np.sort(pd.unique(ABCD_combined[['From Bus Number', 'To Bus Number']].values.ravel()))
for i, bus in enumerate(buses):
    print(f"a{i+1} / b{i+1} -> Bus {bus}")


a1 / b1 -> Bus (2135+0j)
a2 / b2 -> Bus (2220+0j)
a3 / b3 -> Bus (2222+0j)
a4 / b4 -> Bus (2225+0j)
a5 / b5 -> Bus (2230+0j)
a6 / b6 -> Bus (2240+0j)
a7 / b7 -> Bus (2245+0j)
a8 / b8 -> Bus (2250+0j)
a9 / b9 -> Bus (2280+0j)
a10 / b10 -> Bus (2281+0j)
a11 / b11 -> Bus (2300+0j)
a12 / b12 -> Bus (2305+0j)
a13 / b13 -> Bus (2306+0j)
a14 / b14 -> Bus (2307+0j)
a15 / b15 -> Bus (2350+0j)
a16 / b16 -> Bus (2400+0j)
a17 / b17 -> Bus (2560+0j)
a18 / b18 -> Bus (2561+0j)
a19 / b19 -> Bus (2570+0j)
a20 / b20 -> Bus (2580+0j)
a21 / b21 -> Bus (2691+0j)
a22 / b22 -> Bus (2705+0j)
a23 / b23 -> Bus (2810+0j)
a24 / b24 -> Bus (2815+0j)
a25 / b25 -> Bus (2830+0j)
a26 / b26 -> Bus (2970+0j)
